# Vecka 4 – Kapitel 4: kod

Koduppgifterna 11–14 (10 = avskrift av kapitlets exempel). Uppgift 15 ligger i `vecka4_mnist.ipynb`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree

## Uppgift 11 – classification_report

Koden jämför sanna klasser (`y_true`) med modellens gissningar (`y_pred`) för 5 exempel i 3 klasser och skriver ut precision, recall och F1 per klass.

In [ ]:
y_true = [0, 1, 2, 2, 2]
y_pred = [0, 0, 2, 2, 1]
target_names = ['class 0', 'class 1', 'class 2']
print(classification_report(y_true, y_pred, target_names=target_names))

**Tolkning.**
- **class 0**: en sann (index 0) som gissades rätt → recall 1.0. Men modellen gissade "0" två gånger (index 0 rätt, index 1 fel) → precision 1/2 = 0.50.
- **class 1**: en sann (index 1) men modellen gissade 0 där → recall 0 (missade den). Modellen gissade aldrig 1 korrekt → precision 0.
- **class 2**: tre sanna (index 2,3,4). Två gissades rätt, en gissades som 1 → recall 2/3 ≈ 0.67. Alla gånger modellen sa "2" var det rätt → precision 1.0.
- **accuracy** = 3/5 = 0.60 (index 0,2,3 rätt).
- **macro avg** = oförviktat medel över klasserna, **weighted avg** viktar efter antal exempel per klass (support).

## Uppgift 12 – visualisera ett beslutsträd

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
trad = DecisionTreeClassifier(max_depth=3, random_state=42)
trad.fit(iris.data, iris.target)

plt.figure(figsize=(14, 8))
plot_tree(trad, feature_names=iris.feature_names, class_names=iris.target_names,
          filled=True, rounded=True)
plt.show()

Varje ruta visar villkoret som datan delas på, `gini` (orenhet), antal `samples` och klassfördelningen. Färgen visar dominerande klass – ju mer mättad, desto renare nod.

## Uppgift 13 – HR-data, komplett ML-flöde (y = left)

Förutsäg om en anställd har slutat (1) eller stannat (0).

In [ ]:
df = pd.read_excel("../dataset/hr_employee_data.xlsx")

# y = left, X = allt utom target och id. Kategorierna one-hot-kodas.
y = df["left"]
X = df.drop(columns=["left", "Emp_Id"])
X = pd.get_dummies(X, columns=["Department", "salary"], drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print("Andel som slutat (train):", round(y_train.mean(), 3))

In [ ]:
# Jämför två modeller
for namn, m in {"LogReg": LogisticRegression(max_iter=1000),
                "RandomForest": RandomForestClassifier(random_state=42, n_jobs=-1)}.items():
    m.fit(X_train, y_train)
    print(f"\n===== {namn} =====")
    print(classification_report(y_test, m.predict(X_test)))

In [ ]:
# Random forest vann tydligt – titta på confusion matrix + feature importance
rf = RandomForestClassifier(random_state=42, n_jobs=-1).fit(X_train, y_train)

ConfusionMatrixDisplay(confusion_matrix(y_test, rf.predict(X_test))).plot()
plt.title("Random forest – confusion matrix")
plt.show()

vikt = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print(vikt.head(8).round(3))

Random forest klarar detta nästan perfekt. `satisfaction_level`, `time_spend_company` och antal projekt/arbetstimmar är de viktigaste förklaringsvariablerna – lågt nöje driver uppsägning.

## Uppgift 14 – iris, komplett ML-flöde

Given start: två features, train/val/test-split. Vi tränar en modell, väljer utifrån validering och utvärderar på test.

In [ ]:
X, y = load_iris(return_X_y=True, as_frame=True)
X = X[['sepal length (cm)', 'sepal width (cm)']]

X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.3, random_state=42)

classes = ['setosa', 'versicolor', 'virginica']
scatter = plt.scatter(X_train['sepal length (cm)'], X_train['sepal width (cm)'], c=y_train)
plt.xlabel('Sepal Length (cm)'); plt.ylabel('Sepal Width (cm)')
plt.title('Scatter (train)')
plt.legend(handles=scatter.legend_elements()[0], labels=classes)
plt.show()

In [ ]:
# Välj modell på valideringsdatan
for namn, m in {"LogReg": LogisticRegression(max_iter=1000),
                "RandomForest": RandomForestClassifier(random_state=42)}.items():
    m.fit(X_train, y_train)
    print(namn, "val-accuracy:", round(m.score(X_val, y_val), 3))

In [ ]:
# Slutlig utvärdering på testdatan med vald modell
modell = LogisticRegression(max_iter=1000).fit(X_train_full, y_train_full)
print(classification_report(y_test, modell.predict(X_test), target_names=classes))
ConfusionMatrixDisplay(confusion_matrix(y_test, modell.predict(X_test)),
                       display_labels=classes).plot()
plt.show()

Med bara sepal-längd/-bredd är *setosa* lätt att separera medan *versicolor* och *virginica* överlappar – därför hamnar de flesta felen mellan de två. Fler features (petal-måtten) skulle förbättra resultatet.